In [2]:
"""
Program 4 — FedAvg + XGBoost Leaf Embeddings + Neural Classifier
Dataset : ULB Credit Card Fraud (creditcard.csv)

XGBoost is trained centrally as a feature extractor.
Its leaf index embeddings encode rich non-linear feature interactions.
A neural classifier is then trained federatedly on top of these
embeddings using standard FedAvg — combining the best of gradient
boosting (feature power) with federated learning (privacy).

This is the approach used in production FL-fraud systems.
"""

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, roc_curve, f1_score, recall_score
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import copy, warnings
warnings.filterwarnings("ignore")

# ── CONFIG ────────────────────────────────────────────────────────────────────
N_BANKS        = 5
FL_ROUNDS      = 40
LOCAL_EPOCHS   = 5
BATCH_SIZE     = 256
LR             = 0.0005   # lower LR — LayerNorm needs gentler updates than BatchNorm
XGB_TREES      = 100
RANDOM_STATE   = 42
DEVICE         = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# ── LOAD DATA ─────────────────────────────────────────────────────────────────
print("Loading dataset...")
df     = pd.read_csv("/kaggle/input/datasets/organizations/mlg-ulb/creditcardfraud/creditcard.csv")
X_raw  = df.drop("Class", axis=1).values
y      = df["Class"].values
scaler = StandardScaler()
X_raw  = scaler.fit_transform(X_raw)

X_tr, X_te, y_tr, y_te = train_test_split(
    X_raw, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

# ── STEP 1: XGBoost leaf embedding (feature enrichment) ──────────────────────
print("\nTraining XGBoost feature extractor on full training data...")
xgb_model = xgb.XGBClassifier(
    n_estimators=XGB_TREES, max_depth=6, learning_rate=0.05,
    scale_pos_weight=(y_tr == 0).sum() / max((y_tr == 1).sum(), 1),
    use_label_encoder=False, eval_metric="auc",
    random_state=RANDOM_STATE, tree_method="hist",
    device="cuda" if torch.cuda.is_available() else "cpu"
)
xgb_model.fit(X_tr, y_tr)

# Get leaf index embeddings — each tree outputs a leaf ID
# This gives a rich non-linear feature space
def xgb_embed(X):
    leaves = xgb_model.get_booster().predict(
        xgb.DMatrix(X), pred_leaf=True
    ).astype(np.float32)
    # Normalise leaf indices to [0,1]
    leaves = leaves / XGB_TREES
    return leaves

print("Extracting XGBoost leaf embeddings...")
X_tr_emb = xgb_embed(X_tr)
X_te_emb = xgb_embed(X_te)
EMB_DIM  = X_tr_emb.shape[1]
print(f"  Embedding dimension: {EMB_DIM}")

# ── NON-IID SPLIT ─────────────────────────────────────────────────────────────
def non_iid_split(X, y, n, seed=42):
    rng        = np.random.default_rng(seed)
    fi         = np.where(y == 1)[0]
    li         = np.where(y == 0)[0]
    splits     = np.round(rng.dirichlet(np.ones(n) * 0.4) * len(fi)).astype(int)
    splits[-1] = len(fi) - splits[:-1].sum()
    banks, f0  = [], 0
    lpp        = len(li) // n
    for i in range(n):
        fe  = f0 + splits[i]
        idx = np.concatenate([fi[f0:fe], li[i*lpp:(i+1)*lpp]])
        rng.shuffle(idx)
        banks.append((X[idx], y[idx]))
        f0  = fe
    return banks

print(f"\nNon-IID split across {N_BANKS} banks:")
bank_data = non_iid_split(X_tr_emb, y_tr, N_BANKS)
for i, (xb, yb) in enumerate(bank_data):
    print(f"  Bank {i+1}: {len(xb):>6,} records | fraud {yb.mean()*100:.2f}%")

# ── CLASSIFIER MODEL ─────────────────────────────────────────────────────────
class FraudClassifier(nn.Module):
    """
    Deep classifier on top of XGBoost leaf embeddings.
    No BatchNorm — frozen embeddings cause BN stats to diverge
    across banks under FedAvg aggregation, corrupting the model.
    LayerNorm is used instead as it normalises per-sample not per-batch.
    """
    def __init__(self, in_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 256), nn.LayerNorm(256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 128),   nn.LayerNorm(128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 64),    nn.LayerNorm(64),  nn.ReLU(),
            nn.Linear(64, 1),      nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x).squeeze()

def make_loader(X, y):
    return DataLoader(
        TensorDataset(torch.tensor(X, dtype=torch.float32),
                      torch.tensor(y, dtype=torch.float32)),
        batch_size=BATCH_SIZE, shuffle=True
    )

# ── LOCAL TRAIN ───────────────────────────────────────────────────────────────
def local_train(model, loader):
    model.train()
    optimizer = optim.Adam(model.parameters(), lr=LR)
    criterion = nn.BCELoss()
    for _ in range(LOCAL_EPOCHS):
        for X_b, y_b in loader:
            X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(X_b), y_b)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
    return model

# ── EVALUATE ─────────────────────────────────────────────────────────────────
def evaluate(model, X, y):
    model.eval()
    with torch.no_grad():
        prob = model(torch.tensor(X, dtype=torch.float32).to(DEVICE)).cpu().numpy()
    pred  = (prob >= 0.5).astype(int)
    auc   = roc_auc_score(y, prob)
    f1    = f1_score(y, pred, zero_division=0)
    fpr, tpr, _ = roc_curve(y, prob)
    idx   = np.where(fpr <= 0.01)[0]
    r1fpr = tpr[idx[-1]] if len(idx) else 0.0
    K     = max(1, int(len(y) * 0.005))
    pk    = y[np.argsort(prob)[::-1][:K]].mean()
    return dict(auc=auc, f1=f1, recall_1fpr=r1fpr, prec_at_k=pk)

# ── FEDERATED TRAINING ────────────────────────────────────────────────────────
print(f"\nStarting FedAvg + XGBoost Embeddings: {FL_ROUNDS} rounds\n")
global_model = FraudClassifier(EMB_DIM).to(DEVICE)

for rnd in range(1, FL_ROUNDS + 1):
    local_weights = []
    local_sizes   = []

    for bank_id, (X_b, y_b) in enumerate(bank_data):
        try:
            sm = SMOTE(random_state=RANDOM_STATE, k_neighbors=min(3, int(y_b.sum())-1))
            X_r, y_r = sm.fit_resample(X_b, y_b)
        except Exception:
            X_r, y_r = X_b, y_b

        local_model = copy.deepcopy(global_model)
        loader      = make_loader(X_r, y_r)
        local_model = local_train(local_model, loader)

        local_weights.append(copy.deepcopy(local_model.state_dict()))
        local_sizes.append(len(X_b))

    # FedAvg aggregation
    total     = sum(local_sizes)
    new_state = copy.deepcopy(local_weights[0])
    for key in new_state:
        new_state[key] = sum(
            local_weights[i][key] * (local_sizes[i] / total)
            for i in range(N_BANKS)
        )
    global_model.load_state_dict(new_state)

    if rnd % 5 == 0 or rnd == 1:
        m = evaluate(global_model, X_te_emb, y_te)
        print(f"  Round {rnd:>3} | AUC: {m['auc']:.4f} | F1: {m['f1']:.4f} "
              f"| Recall@1%FPR: {m['recall_1fpr']:.4f} | P@K: {m['prec_at_k']:.4f}")

# ── FINAL RESULTS ─────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("FINAL FedAvg + XGBoost Embedding RESULTS")
print("="*60)
m = evaluate(global_model, X_te_emb, y_te)
for k, v in m.items():
    print(f"  {k:<20}: {v:.4f}")

print("\nPer-Bank AUC (Fairness):")
bank_aucs = []
for i, (X_b, y_b) in enumerate(bank_data):
    if len(np.unique(y_b)) < 2: continue
    mb = evaluate(global_model, X_b, y_b)
    bank_aucs.append(mb['auc'])
    print(f"  Bank {i+1}: AUC={mb['auc']:.4f} | Recall@1%FPR={mb['recall_1fpr']:.4f}")
print(f"\n  σ_AUC = {np.std(bank_aucs):.4f}")
print(f"  Jain Fairness Index = {sum(bank_aucs)**2 / (len(bank_aucs)*sum(a**2 for a in bank_aucs)):.4f}")

Device: cuda
Loading dataset...

Training XGBoost feature extractor on full training data...
Extracting XGBoost leaf embeddings...
  Embedding dimension: 100

Non-IID split across 5 banks:
  Bank 1: 45,599 records | fraud 0.24%
  Bank 2: 45,672 records | fraud 0.40%
  Bank 3: 45,490 records | fraud 0.00%
  Bank 4: 45,592 records | fraud 0.22%
  Bank 5: 45,491 records | fraud 0.00%

Starting FedAvg + XGBoost Embeddings: 40 rounds

  Round   1 | AUC: 0.2327 | F1: 0.0000 | Recall@1%FPR: 0.0102 | P@K: 0.0035
  Round   5 | AUC: 0.9494 | F1: 0.8144 | Recall@1%FPR: 0.8571 | P@K: 0.2923
  Round  10 | AUC: 0.9299 | F1: 0.7937 | Recall@1%FPR: 0.8469 | P@K: 0.2923
  Round  15 | AUC: 0.9420 | F1: 0.8083 | Recall@1%FPR: 0.8571 | P@K: 0.2852
  Round  20 | AUC: 0.9481 | F1: 0.7938 | Recall@1%FPR: 0.8469 | P@K: 0.2817
  Round  25 | AUC: 0.9398 | F1: 0.8000 | Recall@1%FPR: 0.8571 | P@K: 0.2817
  Round  30 | AUC: 0.9427 | F1: 0.8208 | Recall@1%FPR: 0.8265 | P@K: 0.2852
  Round  35 | AUC: 0.9418 | F1: 0.